# Architectural Ablations on a Small Transformer
## RoPE vs Learned Positional Embeddings on RF/Telecom Text

**Author:** Muhammad Hanzala Iqbal

**Research question:** Does Rotary Position Embedding (RoPE) outperform a learned positional embedding on a ~14M-parameter decoder-only transformer trained from scratch on a small RF/telecom corpus, measured by validation perplexity and qualitative generation?

**Corpus:** arXiv `eess.SP` abstracts. **MLP activation:** GELU (fixed). **Mixed precision:** fp16 + GradScaler (Tesla T4 is Turing, no bf16 acceleration). **Tracking:** Weights & Biases (public project `rf-rope-ablation`).

**To skip retraining on Run All:** attach a successful prior Commit Version's output as an Input (Add Input → Notebook Output Files → pick a Version). The bootstrap cell below copies the cached artifacts into `/kaggle/working/`; every training cell then skips automatically.

## Section 1 — Setup & Imports

In [ ]:
# Cell 1 - Dependencies
# Kaggle ships a co-tuned scientific stack (CUDA-matched torch + tokenizers /
# datasets / huggingface-hub / transformers). Re-pinning libraries deep in that
# graph drags the whole HF stack backwards and causes resolver conflicts. So:
# install ONLY genuinely-missing leaf packages (wandb, einops).
%pip install -q "wandb>=0.18" "einops>=0.8"

In [ ]:
# Cell 2 - Imports and environment sanity check
import os
import math
import time
import random

import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F

print(f"torch          : {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"GPU count      : {torch.cuda.device_count()}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device         : {device}")

In [ ]:
# Cell 3 - Reproducibility and precision choice
SEED = 1337

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# Tesla T4 = Turing (SM 7.5): FP16 tensor cores, no bf16 acceleration
# (bf16 needs Ampere SM 8.0+). So mixed-precision dtype = float16.
amp_dtype = torch.float16

print(f"seed      : {SEED}")
print(f"amp dtype : {amp_dtype}")

In [ ]:
# Cell 3b - Bootstrap from a prior Commit's saved output (if attached as Input).
# Searches /kaggle/input/ RECURSIVELY for our artifacts and OVERWRITES the
# copy in /kaggle/working (the input is the trusted source).
import shutil
from pathlib import Path

WORKING = Path("/kaggle/working")
WORKING.mkdir(exist_ok=True)
TARGETS = (
    "eess_sp_corpus.txt", "rf_bpe_tokenizer.json",
    "train.bin", "val.bin",
    "baseline-learned-pe.pt", "rope.pt",
)

found = {}
input_root = Path("/kaggle/input")
if input_root.exists():
    for p in input_root.rglob("*"):
        if p.is_file() and p.name in TARGETS and p.name not in found:
            found[p.name] = p

if not found:
    print("[bootstrap] no prior Commit output attached; everything will build fresh.")
else:
    print(f"[bootstrap] found {len(found)} prior artifact(s); OVERWRITING into /kaggle/working")
    for name, src in found.items():
        dst = WORKING / name
        action = "overwrote" if dst.exists() else "copied"
        shutil.copy(src, dst)
        print(f"  {action} {name}  ({src.stat().st_size/1e6:.1f} MB)  from {src}")
    print("[bootstrap] done - using prior Commit's artifacts as ground truth.")

## Section 2 — The Corpus (arXiv eess.SP)

Frozen Kaggle-hosted snapshot of all arXiv metadata, filtered to the `eess.SP` (Signal Processing) category. Cache-or-build: rebuilds only on a fresh session with no prior output attached.

In [ ]:
# Cell 4 - Locate the raw arXiv metadata snapshot
# kagglehub is Kaggle's official downloader: it version-pins the dataset and
# RETURNS the local directory - so we never hardcode a fragile mount path.
import json
import os
from pathlib import Path

import kagglehub

dataset_dir = Path(kagglehub.dataset_download("Cornell-University/arxiv"))
print(f"dataset dir : {dataset_dir}")
print(f"contents    : {os.listdir(dataset_dir)}")

json_files = sorted(dataset_dir.glob("*.json"))
assert json_files, f"No .json snapshot found in {dataset_dir}"
ARXIV_JSON = json_files[0]

size_gb = ARXIV_JSON.stat().st_size / 1e9
print(f"raw snapshot : {ARXIV_JSON}")
print(f"size         : {size_gb:.2f} GB")

In [ ]:
# Cell 5 - Filter to eess.SP and build the cleaned text corpus (cache-or-build)
import re

CORPUS_PATH = Path("/kaggle/working/eess_sp_corpus.txt")
TARGET_CATEGORY = "eess.SP"

if CORPUS_PATH.exists():
    n_lines = sum(1 for _ in CORPUS_PATH.open(encoding="utf-8"))
    mb = CORPUS_PATH.stat().st_size / 1e6
    print(f"[cached] corpus exists: {CORPUS_PATH}  ({n_lines:,} lines, {mb:.1f} MB) - rebuild skipped")
else:
    _ws = re.compile(r"\s+")

    def clean_abstract(text: str) -> str:
        return _ws.sub(" ", text).strip()

    n_total = n_kept = n_chars = 0
    with ARXIV_JSON.open() as fin, CORPUS_PATH.open("w", encoding="utf-8") as fout:
        for line in fin:
            n_total += 1
            rec = json.loads(line)
            if TARGET_CATEGORY in rec["categories"].split():
                abstract = clean_abstract(rec["abstract"])
                fout.write(abstract + "\n")
                n_kept += 1
                n_chars += len(abstract)
    print(f"[built] scanned {n_total:,} | kept {n_kept:,} | {n_chars/1e6:.1f} MB -> {CORPUS_PATH}")

## Section 3 — BPE Tokenizer

Custom byte-level BPE on the RF corpus, vocab 8000. Byte-level => no `<unk>` even on math symbols. Domain-trained => RF terms (OFDM, MIMO) are single tokens, so the 256-token window holds more real content.

In [ ]:
# Cell 6 - Train a custom byte-level BPE tokenizer (cache-or-build)
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

VOCAB_SIZE = 8000
SPECIAL_TOKENS = ["<|endoftext|>"]
TOKENIZER_PATH = Path("/kaggle/working/rf_bpe_tokenizer.json")

if TOKENIZER_PATH.exists():
    print(f"[cached] tokenizer exists: {TOKENIZER_PATH} - retrain skipped")
else:
    tokenizer = Tokenizer(models.BPE(unk_token=None))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=VOCAB_SIZE,
        special_tokens=SPECIAL_TOKENS,
        show_progress=False,
    )
    tokenizer.train([str(CORPUS_PATH)], trainer)
    tokenizer.save(str(TOKENIZER_PATH))
    print(f"[built] vocab {tokenizer.get_vocab_size()} -> {TOKENIZER_PATH}")

In [ ]:
# Cell 7 - Load tokenizer, verify round-trip, measure REAL chars/token
loaded_tok = Tokenizer.from_file(str(TOKENIZER_PATH))

sample = "We propose an OFDM-based MIMO channel estimation scheme using deep learning."
enc = loaded_tok.encode(sample)
roundtrip = loaded_tok.decode(enc.ids)

print(f"sample        : {sample}")
print(f"n tokens      : {len(enc.ids)}")
print(f"round-trip OK : {roundtrip == sample}")

lines = CORPUS_PATH.read_text(encoding="utf-8").splitlines()
encodings = loaded_tok.encode_batch(lines)
n_tokens = sum(len(e.ids) for e in encodings)
n_chars = sum(len(l) for l in lines)
print(f"corpus chars  : {n_chars:,}")
print(f"corpus tokens : {n_tokens:,}")
print(f"chars / token : {n_chars / n_tokens:.2f}")

## Section 4 — Dataset & DataLoader

Encode the corpus to one token stream (abstracts separated by `<|endoftext|>`), deterministic train/val split, persisted as `uint16` `.bin` files. `get_batch` samples random `(batch_size, block_size)` chunks for next-token prediction.

In [ ]:
# Cell 8 - Encode corpus into token stream + reproducible split (cache-or-build)
import numpy as np

TRAIN_BIN = Path("/kaggle/working/train.bin")
VAL_BIN = Path("/kaggle/working/val.bin")
VAL_FRACTION = 0.05
EOT_ID = loaded_tok.token_to_id("<|endoftext|>")

if TRAIN_BIN.exists() and VAL_BIN.exists():
    print("[cached] token bins exist - rebuild skipped")
else:
    lines = CORPUS_PATH.read_text(encoding="utf-8").splitlines()
    encodings = loaded_tok.encode_batch(lines)
    ids = []
    for e in encodings:
        ids.extend(e.ids)
        ids.append(EOT_ID)
    ids = np.array(ids, dtype=np.uint16)
    n_val = int(len(ids) * VAL_FRACTION)
    ids[:-n_val].tofile(TRAIN_BIN)
    ids[-n_val:].tofile(VAL_BIN)
    print(f"[built] {len(ids):,} tokens written -> .bin")

# Summary derived from FILES (works in both branches):
n_train = TRAIN_BIN.stat().st_size // 2
n_val_tok = VAL_BIN.stat().st_size // 2
n_total = n_train + n_val_tok
print(f"EOT id        : {EOT_ID}")
print(f"total tokens  : {n_total:,}")
print(f"train tokens  : {n_train:,}")
print(f"val tokens    : {n_val_tok:,}")
print(f"val fraction  : {n_val_tok / n_total:.3%}")
print(f"dtype / size  : uint16, {n_total * 2 / 1e6:.1f} MB total")

In [ ]:
# Cell 9 - get_batch: random (batch_size, block_size) next-token batches
BATCH_SIZE = 32
BLOCK_SIZE = 256

train_data = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_data = np.memmap(VAL_BIN, dtype=np.uint16, mode="r")

def get_batch(split: str):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([torch.from_numpy(data[i : i + BLOCK_SIZE].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i + 1 : i + 1 + BLOCK_SIZE].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print(f"x shape       : {tuple(xb.shape)}   (batch_size, block_size)")
print(f"x value range : [{xb.min().item()}, {xb.max().item()}]   (token ids in vocab 8000)")
print(f"y == x shifted by 1 ? {torch.equal(xb[0,1:], yb[0,:-1])}")

## Section 5 — The Model

A small decoder-only transformer built in pieces. The `pos_encoding` config field is the **single ablation switch** — `"learned"` (baseline) vs `"rope"` (variant). Everything else is identical between the two runs.

In [ ]:
# Cell 10 - Model configuration
from dataclasses import dataclass

@dataclass
class ModelConfig:
    vocab_size: int = 8000
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384
    dropout: float = 0.1
    pos_encoding: str = "learned"   # "learned" or "rope"  <- THE ablation switch

    def __post_init__(self):
        assert self.n_embd % self.n_head == 0, (
            f"n_embd ({self.n_embd}) must be divisible by n_head ({self.n_head})"
        )

cfg = ModelConfig()
print(cfg)
print(f"head dim = n_embd / n_head = {cfg.n_embd // cfg.n_head}")

In [ ]:
# Cell 11 - Token + positional embeddings (the pos_encoding switch, first use)
class Embeddings(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        if cfg.pos_encoding == "learned":
            self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)
        elif cfg.pos_encoding == "rope":
            self.pos_emb = None
        else:
            raise ValueError(f"unknown pos_encoding: {cfg.pos_encoding}")
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        x = self.tok_emb(idx)
        if self.pos_emb is not None:
            pos = torch.arange(T, device=idx.device)
            x = x + self.pos_emb(pos)
        return self.drop(x)

In [ ]:
# Cell 12 - Causal self-attention WITH optional RoPE (the clean swap)
def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(q, k, cos, sin):
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    return (q * cos + rotate_half(q) * sin,
            k * cos + rotate_half(k) * sin)

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.n_head = cfg.n_head
        self.n_embd = cfg.n_embd
        self.use_rope = (cfg.pos_encoding == "rope")
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size))
        self.register_buffer("mask", mask.view(1, 1, cfg.block_size, cfg.block_size))

        if self.use_rope:
            head_dim = cfg.n_embd // cfg.n_head
            assert head_dim % 2 == 0, "RoPE needs an even head_dim"
            inv_freq = 1.0 / (10000.0 ** (torch.arange(0, head_dim, 2).float() / head_dim))
            t = torch.arange(cfg.block_size).float()
            freqs = torch.outer(t, inv_freq)
            emb = torch.cat([freqs, freqs], dim=-1)
            self.register_buffer("rope_cos", emb.cos())   # buffer => 0 learned params
            self.register_buffer("rope_sin", emb.sin())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(self.n_embd, dim=2)
        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)

        if self.use_rope:                       # <-- the ENTIRE swap is these two lines
            q, k = apply_rope(q, k, self.rope_cos[:T], self.rope_sin[:T])

        att = (q @ k.transpose(-2, -1)) / math.sqrt(head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(y))

In [ ]:
# Cell 13 - MLP + Block (pre-LN + residuals)
class MLP(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=False)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.drop(self.proj(F.gelu(self.fc(x))))

class Block(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

In [ ]:
# Cell 14 - The full GPT model (with proper GPT-2 weight init)
class GPT(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.emb = Embeddings(cfg)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.emb.tok_emb.weight   # tied input/output embeddings

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor = None):
        x = self.emb(idx)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

# sanity check: init loss should be ~ ln(vocab_size) = ~8.99
_model = GPT(cfg).to(device)
_xb, _yb = get_batch("train")
_logits, _loss = _model(_xb, _yb)
print(f"logits : {tuple(_logits.shape)}   (B, T, vocab_size)")
print(f"loss   : {_loss.item():.4f}   (random init -> expect ~ln(8000) = {math.log(8000):.2f})")

In [ ]:
# Cell 15 - Parameter count
def count_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters())

total = count_params(_model)
tok_p = _model.emb.tok_emb.weight.numel()
pos_p = _model.emb.pos_emb.weight.numel() if _model.emb.pos_emb is not None else 0
print(f"token emb (tied w/ lm_head) : {tok_p:,}   = 8000 * 384")
print(f"pos emb (learned)           : {pos_p:,}   = 256 * 384")
print(f"TOTAL parameters            : {total:,}  ({total/1e6:.2f}M)")

## Section 6 — Training Loop

AdamW + linear-warmup → cosine-decay LR + fp16 GradScaler + gradient clipping. Periodically estimate train/val loss, checkpoint on **best val** (not final step), log everything live to Weights & Biases.

In [ ]:
# Cell 16 - Training configuration
@dataclass
class TrainConfig:
    max_steps: int = 5000
    batch_size: int = 32
    block_size: int = 256
    learning_rate: float = 3e-4
    min_lr: float = 3e-5
    warmup_steps: int = 200
    weight_decay: float = 0.1
    grad_clip: float = 1.0
    eval_interval: int = 250
    eval_iters: int = 50
    betas: tuple = (0.9, 0.95)

tcfg = TrainConfig()
print(tcfg)

In [ ]:
# Cell 17 - LR schedule (linear warmup -> cosine decay) and val-loss estimator
def get_lr(step: int, tc: TrainConfig) -> float:
    if step < tc.warmup_steps:
        return tc.learning_rate * (step + 1) / tc.warmup_steps
    if step >= tc.max_steps:
        return tc.min_lr
    ratio = (step - tc.warmup_steps) / (tc.max_steps - tc.warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * ratio))
    return tc.min_lr + coeff * (tc.learning_rate - tc.min_lr)

@torch.no_grad()
def estimate_loss(model: nn.Module, tc: TrainConfig) -> dict:
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(tc.eval_iters)
        for i in range(tc.eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses[i] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

for s in [0, 100, 200, 1000, 2500, 4999]:
    print(f"step {s:5d}  lr = {get_lr(s, tcfg):.2e}")

In [ ]:
# Cell 18 - Weights & Biases setup
# 1. Get your key: https://wandb.ai/authorize
# 2. In Kaggle: Add-ons -> Secrets -> add WANDB_API_KEY = <your key>
import wandb
from kaggle_secrets import UserSecretsClient

wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
WANDB_PROJECT = "rf-rope-ablation"
print("wandb logged in; project:", WANDB_PROJECT)

In [ ]:
# Cell 19 - The training loop (AdamW + fp16 + clip + schedule + eval + W&B + best-val ckpt)
def train_model(model: nn.Module, mcfg: ModelConfig, tc: TrainConfig, run_name: str) -> dict:
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=tc.learning_rate,
        betas=tc.betas, weight_decay=tc.weight_decay,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

    wandb.init(project=WANDB_PROJECT, name=run_name,
               config={**vars(mcfg), **vars(tc)})
    ckpt_path = Path(f"/kaggle/working/{run_name}.pt")
    best_val = float("inf")
    model.train()
    t0 = time.time()

    for step in range(tc.max_steps):
        lr = get_lr(step, tc)
        for g in optimizer.param_groups:
            g["lr"] = lr

        xb, yb = get_batch("train")
        with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=(device == "cuda")):
            _, loss = model(xb, yb)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), tc.grad_clip)
        scaler.step(optimizer)
        scaler.update()

        if step % tc.eval_interval == 0 or step == tc.max_steps - 1:
            losses = estimate_loss(model, tc)
            val_ppl = math.exp(losses["val"])
            tok_per_sec = tc.batch_size * tc.block_size * (step + 1) / (time.time() - t0)
            wandb.log({
                "train_loss": losses["train"], "val_loss": losses["val"],
                "val_ppl": val_ppl, "lr": lr,
                "grad_norm": grad_norm.item(), "tokens_per_sec": tok_per_sec,
            }, step=step)
            print(f"step {step:5d} | train {losses['train']:.3f} | "
                  f"val {losses['val']:.3f} | ppl {val_ppl:7.1f} | lr {lr:.1e}")
            if losses["val"] < best_val:
                best_val = losses["val"]
                torch.save({"model": model.state_dict(), "mcfg": mcfg,
                            "step": step, "val_loss": best_val}, ckpt_path)

    wandb.finish()
    print(f"done. best val loss {best_val:.4f} (ppl {math.exp(best_val):.1f}) -> {ckpt_path}")
    return {"best_val_loss": best_val, "ckpt": str(ckpt_path)}

print("train_model defined")

In [ ]:
# Cell 20 - Smoke test (cache-aware): skip if a real checkpoint already exists.
if Path("/kaggle/working/baseline-learned-pe.pt").exists() or Path("/kaggle/working/rope.pt").exists():
    print("[skip] real checkpoint already exists - smoke test not needed.")
else:
    set_seed(SEED)
    smoke_model = GPT(cfg).to(device)
    smoke_tc = TrainConfig(max_steps=100, warmup_steps=20, eval_interval=25)
    _ = train_model(smoke_model, cfg, smoke_tc, run_name="smoke-test")

In [ ]:
# Cell 21 - Baseline training (cache-or-train): learned positional embedding
BASE_CKPT = Path("/kaggle/working/baseline-learned-pe.pt")
if BASE_CKPT.exists():
    print(f"[cached] {BASE_CKPT.name} exists - baseline training skipped")
    baseline_result = {"ckpt": str(BASE_CKPT)}
else:
    set_seed(SEED)
    baseline_cfg = ModelConfig(pos_encoding="learned")
    baseline_model = GPT(baseline_cfg).to(device)
    baseline_result = train_model(baseline_model, baseline_cfg, tcfg,
                                  run_name="baseline-learned-pe")
print(baseline_result)

## Section 7 — Evaluation & Generation

Load the best-val baseline checkpoint, re-verify perplexity, and generate RF/telecom text with temperature + top-p sampling (temp 0.8, top-p 0.9).

In [ ]:
# Cell 22 - Load the best-val baseline checkpoint and re-verify perplexity
ckpt = torch.load("/kaggle/working/baseline-learned-pe.pt",
                   map_location=device, weights_only=False)
eval_model = GPT(ckpt["mcfg"]).to(device)
eval_model.load_state_dict(ckpt["model"])
eval_model.eval()

losses = estimate_loss(eval_model, tcfg)
print(f"checkpoint step : {ckpt['step']}")
print(f"val loss        : {losses['val']:.4f}")
print(f"val perplexity  : {math.exp(losses['val']):.2f}")

In [ ]:
# Cell 23 - Autoregressive text generation (temperature + top-p / nucleus sampling)
@torch.no_grad()
def generate(model: nn.Module, prompt: str, max_new_tokens: int = 100,
             temperature: float = 0.8, top_p: float = 0.9) -> str:
    model.eval()
    ids = loaded_tok.encode(prompt).ids
    x = torch.tensor([ids], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        x_cond = x[:, -model.cfg.block_size:]
        logits, _ = model(x_cond)
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        sp, si = torch.sort(probs, descending=True)
        keep = (torch.cumsum(sp, dim=-1) - sp) <= top_p
        sp = sp * keep
        sp = sp / sp.sum(dim=-1, keepdim=True)
        nxt = si.gather(-1, torch.multinomial(sp, num_samples=1))
        x = torch.cat([x, nxt], dim=1)
    return loaded_tok.decode(x[0].tolist())

print("generate() defined")

In [ ]:
# Cell 24 - Qualitative baseline generations (5 RF/telecom prompts)
prompts = [
    "We propose a novel",
    "In this paper, a deep learning approach for channel estimation",
    "Orthogonal frequency division multiplexing (OFDM)",
    "The proposed MIMO system",
    "Massive MIMO and beamforming",
]
for i, p in enumerate(prompts, 1):
    print(f"\n=== Prompt {i}: {p!r} ===")
    print(generate(eval_model, p, max_new_tokens=100, temperature=0.8, top_p=0.9))

## Section 8 — RoPE Variant + Head-to-Head

The clean swap: `pos_encoding="rope"` is the **only** change. Train RoPE with the identical seed and recipe, then compare both checkpoints' val PPL and generations side by side.

In [ ]:
# Cell 25 - RoPE training (cache-or-train): same recipe, only pos_encoding flips.
ROPE_CKPT = Path("/kaggle/working/rope.pt")
if ROPE_CKPT.exists():
    print(f"[cached] {ROPE_CKPT.name} exists - RoPE training skipped")
    rope_result = {"ckpt": str(ROPE_CKPT)}
else:
    set_seed(SEED)
    rope_cfg = ModelConfig(pos_encoding="rope")
    rope_model = GPT(rope_cfg).to(device)
    rope_result = train_model(rope_model, rope_cfg, tcfg, run_name="rope")
print(rope_result)

In [ ]:
# Cell 26 - Head-to-head: load both checkpoints, compare val PPL + generations
def load_ckpt(path):
    ck = torch.load(path, map_location=device, weights_only=False)
    m = GPT(ck["mcfg"]).to(device)
    m.load_state_dict(ck["model"])
    m.eval()
    return m, ck

base_model, base_ck = load_ckpt("/kaggle/working/baseline-learned-pe.pt")
rope_model, rope_ck = load_ckpt("/kaggle/working/rope.pt")

base_losses = estimate_loss(base_model, tcfg)
rope_losses = estimate_loss(rope_model, tcfg)
base_ppl = math.exp(base_losses["val"])
rope_ppl = math.exp(rope_losses["val"])

def n_params(m): return sum(p.numel() for p in m.parameters())

print("=" * 60)
print(f"{'metric':<18}  {'baseline (learned)':>20}  {'RoPE':>14}")
print("-" * 60)
print(f"{'params':<18}  {n_params(base_model):>20,}  {n_params(rope_model):>14,}")
print(f"{'best-val step':<18}  {base_ck['step']:>20}  {rope_ck['step']:>14}")
print(f"{'val loss':<18}  {base_losses['val']:>20.4f}  {rope_losses['val']:>14.4f}")
print(f"{'val perplexity':<18}  {base_ppl:>20.2f}  {rope_ppl:>14.2f}")
print("=" * 60)
print(f"RoPE - baseline = {rope_ppl - base_ppl:+.2f} ppl   "
      f"({(base_ppl - rope_ppl)/base_ppl*100:+.1f}% improvement)")

prompts = [
    "We propose a novel",
    "In this paper, a deep learning approach for channel estimation",
    "Orthogonal frequency division multiplexing (OFDM)",
    "The proposed MIMO system",
    "Massive MIMO and beamforming",
]
for i, p in enumerate(prompts, 1):
    print(f"\n========== Prompt {i}: {p!r} ==========")
    print(f"--- BASELINE (learned PE) ---")
    print(generate(base_model, p, max_new_tokens=100, temperature=0.8, top_p=0.9))
    print(f"--- RoPE ---")
    print(generate(rope_model, p, max_new_tokens=100, temperature=0.8, top_p=0.9))

In [ ]:
# Cell 27 - Write results/comparison_table.md (the deliverable)
RESULTS_DIR = Path("/kaggle/working/results")
RESULTS_DIR.mkdir(exist_ok=True)

delta_pct = (base_ppl - rope_ppl) / base_ppl * 100
md = f"""# Results - RoPE vs Learned PE on RF/Telecom Abstracts

| Metric | Baseline (learned PE) | RoPE | Delta |
|---|---:|---:|---:|
| Parameters | {n_params(base_model):,} | {n_params(rope_model):,} | {n_params(rope_model)-n_params(base_model):+,} |
| Best-val step | {base_ck['step']} | {rope_ck['step']} | - |
| Val loss | {base_losses['val']:.4f} | {rope_losses['val']:.4f} | {rope_losses['val']-base_losses['val']:+.4f} |
| **Val perplexity** | **{base_ppl:.2f}** | **{rope_ppl:.2f}** | **{delta_pct:+.1f}%** |

**Setup.** ~13.8M-param decoder-only transformer (6 layers, 6 heads, n_embd=384, block_size=256). Byte-level BPE, vocab 8000, trained on arXiv `eess.SP` abstracts (9.45M training tokens). Same seed, same recipe (5000 steps, batch 32, AdamW lr=3e-4 with 200-step warmup + cosine decay, fp16+GradScaler, dropout 0.1, grad-norm clip 1.0). The **only** difference between runs: `pos_encoding`.

Comparison reported at each model's **best-val** checkpoint (early-stop discipline, not fixed-step).
"""

(RESULTS_DIR / "comparison_table.md").write_text(md)
print(md)
print(f"\nwritten to: {RESULTS_DIR/'comparison_table.md'}")